In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('dirty_dataset.csv')

## 🟢 STEP 1 — Schema check (Task 15 start)

In [18]:
print(df.shape)
print(df.columns)
print(df.isnull().mean())

# drop useless columns
cols_to_drop = df.columns[df.isnull().mean() > 0.5].tolist()
df.drop(columns=cols_to_drop, inplace=True)

(20300, 15)
Index(['employee_id', 'name', 'age', 'salary', 'join_date', 'department',
       'gender', 'country', 'city', 'weight_kg', 'is_active', 'review',
       'price', 'weight_kg_duplicate', 'target'],
      dtype='object')
employee_id            0.000000
name                   0.000000
age                    0.030591
salary                 0.030049
join_date              0.039901
department             0.042217
gender                 0.080394
country                0.029951
city                   0.029754
weight_kg              0.031675
is_active              0.028867
review                 0.072167
price                  0.021970
weight_kg_duplicate    1.000000
target                 0.000000
dtype: float64


## 🟢 STEP 2 — Fix Data Types (Task 05, 04, 12)


In [19]:
# numeric conversion
df["salary"] = df["salary"].str.replace("$", "", regex=False).astype(float)
df["weight_kg"] = df["weight_kg"].str.replace("kg", "", regex=False).astype(float)
df["price"] = df["price"].str.replace("$", "", regex=False).astype(float)

# date conversion
df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce")

## 🟢 STEP 3 — Missing values (Task 01)

In [20]:
num_cols = ["age", "salary", "price", "weight_kg"]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())


cat_cols = ["department", "gender", "country", "city"]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


df["review"] = df["review"].fillna("No Review")
df["is_active"] = df["is_active"].fillna(False)

# join_date fill
df["join_date"] = df["join_date"].fillna(df["join_date"].mode()[0])

## 🟢 STEP 4 — Remove duplicates (Task 02, 03)

In [21]:
df.drop_duplicates(inplace=True)
df = df.drop_duplicates(subset=["employee_id"], keep="first")

## 🟢 STEP 5 — Standardize labels (Task 06)

In [ ]:
# check
df['country'].unique()
df["country"] = df["country"].str.lower().map({
    "usa": "United States",
    "u.s.a": "United States",
    "us": "United States",
    "america": "United States"
}).fillna(df["country"])

df["gender"] = df["gender"].str.lower().map({
    "male": "Male",
    "m": "Male",
    "man": "Male",
    "female": "Female",
    "f": "Female",
    "woman": "Female"
})

df["department"] = df["department"].str.strip().str.title()

## 🟢 STEP 6 — Fix spelling mistakes (Task 07)

In [23]:
city_fix = {
    "Dhka": "Dhaka",
    "Dhakka": "Dhaka",
    "Chitagong": "Chittagong",
    "Chittagongg": "Chittagong",
    "Slyhet": "Sylhet",
    "Rajshai": "Rajshahi"
}

df["city"] = df["city"].replace(city_fix)

df["department"] = df["department"].replace({
    "Mrketing": "Marketing",
    "Finace": "Finance"
})

## 🟢 STEP 7 — Outliers (Task 08)

In [24]:
Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df["salary"] = df["salary"].clip(lower, upper)

## 🟢 STEP 8 — Invalid values (Task 09, 13)

In [25]:
# age range fix
df.loc[(df["age"] < 18) | (df["age"] > 65), "age"] = df["age"].median()

# price negative fix
df.loc[df["price"] < 0, "price"] = df["price"].median()

# range clipping
df["age"] = df["age"].clip(18, 65)
df["salary"] = df["salary"].clip(15000, 500000)
df["price"] = df["price"].clip(0, 10000)

## 🟢 STEP 9 — Noisy text (Task 10)

In [26]:
noise = ["ok", "bad", "na", "n/a", "...", "fine", "not bad", "ok ok", "."]

df["review"] = df["review"].replace(noise, np.nan)
df["review"] = df["review"].fillna("No Review")

## 🟢 STEP 10 — Boolean fix (Task 11)

In [27]:
true_vals = ["true", "1", "True", 1, True]

df["is_active"] = df["is_active"].isin(true_vals)

## 🟢 STEP 11 — Final validation (Task 15 final)

In [28]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.dtypes)
print(df.describe())

employee_id    0
name           0
age            0
salary         0
join_date      0
department     0
gender         0
country        0
city           0
weight_kg      0
is_active      0
review         0
price          0
target         0
dtype: int64
0
employee_id             int64
name                   object
age                   float64
salary                float64
join_date      datetime64[ns]
department             object
gender                 object
country                object
city                   object
weight_kg             float64
is_active                bool
review                 object
price                 float64
target                  int64
dtype: object
        employee_id           age         salary  \
count  19801.000000  19801.000000   19801.000000   
mean   10001.784203     41.443412   75126.790314   
min        1.000000     18.000000   15000.000000   
25%     5001.000000     30.000000   52202.000000   
50%    10008.000000     41.000000   74404.500000   
7

## 🟢 STEP 12 — Class imbalance (Task 14)

In [ ]:
# Check distribution
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True)*100)

# ---------------------
# Oversampling
# ---------------------
from sklearn.utils import resample

majority = df[df.target == 0]
minority = df[df.target == 1]

minority_up = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

df_balanced = pd.concat(
    [majority, minority_up]
)


target
0    18814
1      987
Name: count, dtype: int64
target
0    95.015403
1     4.984597
Name: proportion, dtype: float64
